In [0]:
from pyspark.sql.functions import col

dbutils.widgets.text("batch_id", "full-sample-20261010-01", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()

if not batch_id:
    raise ValueError("batch_id is required")

bronze_batch_df = (
    spark.table("workspace.tmdb_lakehouse.bronze_movie_records")
    .filter(col("batch_id") == batch_id)
)

batch_count = bronze_batch_df.count()
if batch_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

print(f"Bronze rows in selected batch: {batch_count}")
bronze_batch_df.select(
    "batch_id",
    "load_type",
    "source_extracted_at",
    "window_start",
    "window_end",
).distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col, from_json, size
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

movie_schema = StructType([
    StructField("id", LongType(), True),
    StructField("imdb_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("original_title", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("original_language", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("adult", BooleanType(), True),
    StructField("video", BooleanType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("vote_average", DoubleType(), True),
    StructField("vote_count", LongType(), True),
    StructField("budget", LongType(), True),
    StructField("revenue", LongType(), True),

    StructField("genres", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ])), True),

    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),

    StructField("credits", StructType([
        StructField("cast", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("credit_id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("character", StringType(), True),
            StructField("order", IntegerType(), True),
        ])), True),
    ]), True),

    StructField("keywords", StructType([
        StructField("keywords", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("name", StringType(), True),
        ])), True),
    ]), True),

    StructField("reviews", StructType([
        StructField("results", ArrayType(StructType([
            StructField("id", StringType(), True),
            StructField("author", StringType(), True),
            StructField("author_details", StructType([
                StructField("name", StringType(), True),
                StructField("username", StringType(), True),
                StructField("avatar_path", StringType(), True),
                StructField("rating", DoubleType(), True),
            ]), True),
            StructField("content", StringType(), True),
            StructField("created_at", StringType(), True),
            StructField("updated_at", StringType(), True),
        ])), True),
    ]), True),
])

parsed_movies_df = (
    bronze_batch_df
    .select(
        col("movie_id").alias("bronze_movie_id"),
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        from_json(col("raw_payload"), movie_schema).alias("movie"),
    )
    .select(
        "bronze_movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        "movie.*",
    )
)

parsed_movies_df.printSchema()

print("Bronze rows in batch:", bronze_batch_df.count())
print("Parsed movie IDs missing:", parsed_movies_df.filter(col("id").isNull()).count())
print("Parsed titles missing:", parsed_movies_df.filter(col("title").isNull()).count())

display(
    parsed_movies_df.select(
        "bronze_movie_id",
        "id",
        "title",
        "budget",
        "revenue",
        size("genres").alias("genre_count"),
        size("production_companies").alias("studio_count"),
        size("credits.cast").alias("cast_count"),
        size("keywords.keywords").alias("keyword_count"),
        size("reviews.results").alias("review_count"),
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

# Stop rather than silently accepting records that fail the movie contract.
invalid_movies = parsed_movies_df.filter(
    F.col("id").isNull()
    | F.col("title").isNull()
    | (F.col("id") != F.col("bronze_movie_id"))
)

invalid_count = invalid_movies.count()
if invalid_count:
    display(invalid_movies.select("bronze_movie_id", "id", "title").limit(20))
    raise ValueError(
        f"{invalid_count} records have a missing/mismatched movie ID or missing title. "
        "Do not continue until they are quarantined."
    )

clean_budget = F.when(F.col("budget") > 0, F.col("budget")).otherwise(
    F.lit(None).cast(LongType())
)
clean_revenue = F.when(F.col("revenue") > 0, F.col("revenue")).otherwise(
    F.lit(None).cast(LongType())
)

silver_movies_source_df = (
    parsed_movies_df
    .select(
        F.col("id").alias("movie_id"),
        "imdb_id",
        "title",
        "original_title",
        "overview",
        "original_language",
        F.to_date("release_date", "yyyy-MM-dd").alias("release_date"),
        "runtime",
        "status",
        "adult",
        "video",
        "popularity",
        "vote_average",
        "vote_count",
        clean_budget.alias("budget"),
        clean_revenue.alias("revenue"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            clean_revenue - clean_budget,
        ).cast(LongType()).alias("profit"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            (clean_revenue - clean_budget) / clean_budget,
        ).cast("double").alias("roi"),
        F.when(clean_budget.isNull(), F.lit(None).cast("string"))
        .when(clean_budget < 10000000, F.lit("low"))
        .when(clean_budget < 100000000, F.lit("medium"))
        .otherwise(F.lit("high"))
        .alias("budget_tier"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

business_columns = [
    "movie_id",
    "imdb_id",
    "title",
    "original_title",
    "overview",
    "original_language",
    "release_date",
    "runtime",
    "status",
    "adult",
    "video",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
    "profit",
    "roi",
    "budget_tier",
]

silver_movies_source_df = (
    silver_movies_source_df
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct(*[F.col(name) for name in business_columns])),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

print("Silver movie rows prepared:", silver_movies_source_df.count())
silver_movies_source_df.printSchema()

display(
    silver_movies_source_df.select(
        "movie_id",
        "title",
        "release_date",
        "budget",
        "revenue",
        "profit",
        "roi",
        "budget_tier",
        "source_batch_id",
        "source_extracted_at",
    ).limit(10)
)

In [0]:
print(
    "Unknown-budget rows with a tier:",
    silver_movies_source_df.filter(
        F.col("budget").isNull() & F.col("budget_tier").isNotNull()
    ).count()
)

In [0]:
from pyspark.sql import functions as F

duplicate_movie_keys = (
    silver_movies_source_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_financials = silver_movies_source_df.filter(
    (
        F.col("budget").isNotNull()
        & F.col("revenue").isNotNull()
        & (
            (F.col("profit") != F.col("revenue") - F.col("budget"))
            | (
                F.abs(
                    F.col("roi")
                    - ((F.col("revenue") - F.col("budget")) / F.col("budget"))
                ) > 1e-9
            )
        )
    )
    | (
        (F.col("budget").isNull() | F.col("revenue").isNull())
        & (F.col("profit").isNotNull() | F.col("roi").isNotNull())
    )
).count()

invalid_budget_tiers = silver_movies_source_df.filter(
    (F.col("budget").isNull() & F.col("budget_tier").isNotNull())
    | (
        (F.col("budget") > 0)
        & (F.col("budget") < 10000000)
        & (F.col("budget_tier") != "low")
    )
    | (
        (F.col("budget") >= 10000000)
        & (F.col("budget") < 100000000)
        & (F.col("budget_tier") != "medium")
    )
    | (
        (F.col("budget") >= 100000000)
        & (F.col("budget_tier") != "high")
    )
).count()

missing_hashes = silver_movies_source_df.filter(
    F.col("row_hash").isNull()
).count()

print("Rows prepared:", silver_movies_source_df.count())
print("Duplicate movie keys:", duplicate_movie_keys)
print("Invalid financial calculations:", invalid_financials)
print("Invalid budget tiers:", invalid_budget_tiers)
print("Missing row hashes:", missing_hashes)

if any([duplicate_movie_keys, invalid_financials, invalid_budget_tiers, missing_hashes]):
    raise ValueError("Silver movie validation failed; do not write the table.")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movies_cleansed (
    movie_id BIGINT,
    imdb_id STRING,
    title STRING,
    original_title STRING,
    overview STRING,
    original_language STRING,
    release_date DATE,
    runtime INT,
    status STRING,
    adult BOOLEAN,
    video BOOLEAN,
    popularity DOUBLE,
    vote_average DOUBLE,
    vote_count BIGINT,
    budget BIGINT,
    revenue BIGINT,
    profit BIGINT,
    roi DOUBLE,
    budget_tier STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movies_cleansed").show(truncate=False)

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

silver_table = "workspace.tmdb_lakehouse.silver_movies_cleansed"
logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"

run_id = str(uuid4())
run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

rows_inserted = 0
rows_updated = 0

def write_silver_run_log(status, error_details=None):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    log_row = [(
        run_id,
        "Bronze-to-Silver",
        bronze_batch_df.select("load_type").first()["load_type"],
        f"batch_id={batch_id}",
        batch_id,
        run_started_at,
        finished_at,
        status,
        rows_inserted,
        rows_updated,
        0,
        error_details,
        finished_at,
    )]

    (
        spark.createDataFrame(log_row, schema=log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(logs_table)
    )

try:
    # A source snapshot wins only if it is newer, or its batch ID is
    # lexically later when two snapshots have the same extraction time.
    source = silver_movies_source_df.alias("source")
    target = spark.table(silver_table).alias("target")

    newer_snapshot = (
        F.col("target.source_extracted_at").isNull()
        | (F.col("source.source_extracted_at") > F.col("target.source_extracted_at"))
        | (
            (F.col("source.source_extracted_at") == F.col("target.source_extracted_at"))
            & (F.col("source.source_batch_id") > F.col("target.source_batch_id"))
        )
    )

    rows_inserted = (
        source.join(
            target,
            F.col("source.movie_id") == F.col("target.movie_id"),
            "left_anti",
        )
        .count()
    )

    rows_updated = (
        source.join(
            target,
            F.col("source.movie_id") == F.col("target.movie_id"),
            "inner",
        )
        .filter(newer_snapshot)
        .count()
    )

    silver_movies_source_df.createOrReplaceTempView("silver_movie_merge_source")

    spark.sql(f"""
        MERGE INTO {silver_table} AS target
        USING silver_movie_merge_source AS source
        ON target.movie_id = source.movie_id

        WHEN MATCHED AND (
            target.source_extracted_at IS NULL
            OR source.source_extracted_at > target.source_extracted_at
            OR (
                source.source_extracted_at = target.source_extracted_at
                AND source.source_batch_id > target.source_batch_id
            )
        )
        THEN UPDATE SET
            target.imdb_id = source.imdb_id,
            target.title = source.title,
            target.original_title = source.original_title,
            target.overview = source.overview,
            target.original_language = source.original_language,
            target.release_date = source.release_date,
            target.runtime = source.runtime,
            target.status = source.status,
            target.adult = source.adult,
            target.video = source.video,
            target.popularity = source.popularity,
            target.vote_average = source.vote_average,
            target.vote_count = source.vote_count,
            target.budget = source.budget,
            target.revenue = source.revenue,
            target.profit = source.profit,
            target.roi = source.roi,
            target.budget_tier = source.budget_tier,
            target.source_batch_id = source.source_batch_id,
            target.source_file = source.source_file,
            target.source_extracted_at = source.source_extracted_at,
            target.row_hash = source.row_hash,
            target.load_timestamp = CASE
                WHEN NOT (target.row_hash <=> source.row_hash)
                THEN source.load_timestamp
                ELSE target.load_timestamp
            END

        WHEN NOT MATCHED THEN INSERT (
            movie_id, imdb_id, title, original_title, overview,
            original_language, release_date, runtime, status, adult, video,
            popularity, vote_average, vote_count, budget, revenue, profit,
            roi, budget_tier, source_batch_id, source_file,
            source_extracted_at, row_hash, load_timestamp
        ) VALUES (
            source.movie_id, source.imdb_id, source.title,
            source.original_title, source.overview, source.original_language,
            source.release_date, source.runtime, source.status, source.adult,
            source.video, source.popularity, source.vote_average,
            source.vote_count, source.budget, source.revenue, source.profit,
            source.roi, source.budget_tier, source.source_batch_id,
            source.source_file, source.source_extracted_at, source.row_hash,
            source.load_timestamp
        )
    """)

    actual_count = (
        spark.table(silver_table)
        .filter(F.col("source_batch_id") == batch_id)
        .count()
    )
    expected_count = silver_movies_source_df.count()

    if actual_count != expected_count:
        raise RuntimeError(
            f"Expected {expected_count} Silver rows from this batch, "
            f"found {actual_count}"
        )

    write_silver_run_log("SUCCESS")
    print(f"Bronze-to-Silver merge succeeded. Rows inserted: {rows_inserted}; rows updated: {rows_updated}")
    print(f"Rows in Silver from this batch: {actual_count}")

except Exception as exc:
    try:
        write_silver_run_log("FAILURE", str(exc)[:4000])
    except Exception as log_exc:
        print(f"Could not write the failure log: {log_exc}")
    raise

In [0]:
from pyspark.sql import functions as F

valid_genre_movies = parsed_movies_df.filter(F.col("genres").isNotNull())

silver_genres_source_df = (
    valid_genre_movies
    .select(
        "bronze_movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("genres").alias("genre"),
    )
    .select(
        F.col("bronze_movie_id").alias("movie_id"),
        F.col("genre.id").alias("genre_id"),
        F.col("genre.name").alias("genre_name"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_genres = silver_genres_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("genre_id").isNull()
    | F.col("genre_name").isNull()
)
invalid_genre_count = invalid_genres.count()

if invalid_genre_count:
    display(invalid_genres.limit(20))
    raise ValueError(f"{invalid_genre_count} genre rows have missing required fields")

silver_genres_source_df = silver_genres_source_df.dropDuplicates(
    ["movie_id", "genre_id"]
)

silver_genres_source_df = (
    silver_genres_source_df
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct("movie_id", "genre_id", "genre_name")),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_genre_keys = (
    silver_genres_source_df
    .groupBy("movie_id", "genre_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a genre collection:", valid_genre_movies.count())
print("Silver genre rows prepared:", silver_genres_source_df.count())
print("Duplicate movie-genre keys:", duplicate_genre_keys)

silver_genres_source_df.printSchema()
display(silver_genres_source_df.limit(10))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_genres (
    movie_id BIGINT,
    genre_id BIGINT,
    genre_name STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_genres
""").show(truncate=False)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_child_snapshot_state (
    movie_id BIGINT,
    collection_name STRING,
    source_extracted_at TIMESTAMP,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_child_snapshot_state
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

genre_snapshot_candidates_df = (
    parsed_movies_df
    .filter(F.col("genres").isNotNull())
    .select(
        F.col("id").alias("movie_id"),
        F.col("source_extracted_at"),
        F.col("batch_id").alias("source_batch_id"),
        F.size("genres").alias("genre_count"),
    )
    .dropDuplicates(["movie_id"])
)

missing_genre_collections = parsed_movies_df.filter(F.col("genres").isNull()).count()
valid_genre_collections = genre_snapshot_candidates_df.count()
empty_genre_collections = genre_snapshot_candidates_df.filter(
    F.col("genre_count") == 0
).count()

print("Movies with a present genre collection:", valid_genre_collections)
print("Movies with an empty genre collection:", empty_genre_collections)
print("Movies with a missing or unparsed genre collection:", missing_genre_collections)

genre_snapshot_candidates_df.printSchema()
display(genre_snapshot_candidates_df.limit(10))

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

genre_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

def log_genre_run(
    run_id,
    started_at,
    status,
    inserted,
    updated,
    quarantined,
    error_details=None,
):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = bronze_batch_df.select("load_type").first()["load_type"]

    row = [(
        run_id,
        "Bronze-to-Silver",
        load_type,
        f"batch_id={batch_id}; table=silver_movie_genres",
        batch_id,
        started_at,
        finished_at,
        status,
        inserted,
        updated,
        quarantined,
        error_details,
        finished_at,
    )]

    (
        spark.createDataFrame(row, schema=genre_log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable("workspace.tmdb_lakehouse.pipeline_execution_logs")
    )

In [0]:
from pyspark.sql import functions as F

genres_table = "workspace.tmdb_lakehouse.silver_movie_genres"
state_table = "workspace.tmdb_lakehouse.silver_child_snapshot_state"

genre_run_id = str(uuid4())
genre_run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
rows_inserted = 0
rows_updated = 0
rows_quarantined = 0

try:
    # Only present, successfully parsed collections are eligible.
    # A present empty array is included here with genre_count = 0.
    state = (
        spark.table(state_table)
        .filter(F.col("collection_name") == "genres")
        .select(
            "movie_id",
            F.col("source_extracted_at").alias("stored_extracted_at"),
            F.col("source_batch_id").alias("stored_batch_id"),
        )
    )

except Exception as exc:
    try:
        log_genre_run(
            genre_run_id,
            genre_run_started_at,
            "FAILURE",
            rows_inserted,
            rows_updated,
            rows_quarantined,
            str(exc)[:4000],
        )
    except Exception as log_exc:
        raise RuntimeError(
            f"Genre processing failed: {exc}; failure logging also failed: {log_exc}"
        ) from log_exc
    raise
else:
    log_genre_run(
        genre_run_id,
        genre_run_started_at,
        "SUCCESS",
        rows_inserted,
        rows_updated,
        rows_quarantined,
    )

eligible_genre_snapshots_df = (
    genre_snapshot_candidates_df.alias("incoming")
    .join(state.alias("stored"), on="movie_id", how="left")
    .filter(
        F.col("stored.stored_batch_id").isNull()
        | (F.col("incoming.source_extracted_at") > F.col("stored.stored_extracted_at"))
        | (
            (F.col("incoming.source_extracted_at") == F.col("stored.stored_extracted_at"))
            & (F.col("incoming.source_batch_id") == F.col("stored.stored_batch_id"))
        )
    )
    .select(
        "movie_id",
        "source_extracted_at",
        "source_batch_id",
        "genre_count",
    )
)

eligible_count = eligible_genre_snapshots_df.count()
print("Eligible movie genre snapshots:", eligible_count)

# Do not proceed if any input snapshot was silently rejected.
if eligible_count != genre_snapshot_candidates_df.count():
    raise ValueError(
        "At least one genre snapshot is not newer than the stored snapshot "
        "and is not a retry of the same batch. Resolve its source timestamp "
        "before processing."
    )

eligible_ids_df = eligible_genre_snapshots_df.select("movie_id").distinct()

eligible_genres_df = (
    silver_genres_source_df
    .join(eligible_ids_df, on="movie_id", how="inner")
)

# Calculate merge metrics before modifying the target.
existing_genres_df = spark.table(genres_table)

rows_inserted = (
    eligible_genres_df
    .join(existing_genres_df, ["movie_id", "genre_id"], "left_anti")
    .count()
)

rows_updated = (
    eligible_genres_df.alias("incoming")
    .join(
        existing_genres_df.alias("stored"),
        (F.col("incoming.movie_id") == F.col("stored.movie_id"))
        & (F.col("incoming.genre_id") == F.col("stored.genre_id")),
        "inner",
    )
    .filter(
        (~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash")))
        | (~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at")))
        | (~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id")))
        | (~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file")))
    )
    .count()
)

# Find prior genre rows that are absent from the complete incoming collection.
# This also finds every old row for a valid empty collection.
stale_genres_df = (
    existing_genres_df
    .join(eligible_ids_df, on="movie_id", how="inner")
    .join(
        eligible_genres_df.select("movie_id", "genre_id"),
        on=["movie_id", "genre_id"],
        how="left_anti",
    )
    .select("movie_id", "genre_id")
    .dropDuplicates()
)

eligible_genre_snapshots_df.createOrReplaceTempView("eligible_genre_snapshots")
eligible_genres_df.createOrReplaceTempView("eligible_genres_source")
stale_genres_df.createOrReplaceTempView("stale_genres_to_delete")

spark.sql(f"""
    MERGE INTO {genres_table} AS target
    USING eligible_genres_source AS source
    ON target.movie_id = source.movie_id
       AND target.genre_id = source.genre_id

    WHEN MATCHED AND (
        NOT (target.row_hash <=> source.row_hash)
        OR NOT (target.source_extracted_at <=> source.source_extracted_at)
        OR NOT (target.source_batch_id <=> source.source_batch_id)
        OR NOT (target.source_file <=> source.source_file)
    )
    THEN UPDATE SET
        target.genre_name = source.genre_name,
        target.source_batch_id = source.source_batch_id,
        target.source_file = source.source_file,
        target.source_extracted_at = source.source_extracted_at,
        target.row_hash = source.row_hash,
        target.load_timestamp = CASE
            WHEN NOT (target.row_hash <=> source.row_hash)
            THEN source.load_timestamp
            ELSE target.load_timestamp
        END

    WHEN NOT MATCHED THEN INSERT (
        movie_id,
        genre_id,
        genre_name,
        source_batch_id,
        source_file,
        source_extracted_at,
        row_hash,
        load_timestamp
    ) VALUES (
        source.movie_id,
        source.genre_id,
        source.genre_name,
        source.source_batch_id,
        source.source_file,
        source.source_extracted_at,
        source.row_hash,
        source.load_timestamp
    )
""")

# Delete stale keys only for movies with eligible, complete collections.
spark.sql(f"""
    MERGE INTO {genres_table} AS target
    USING stale_genres_to_delete AS stale
    ON target.movie_id = stale.movie_id
       AND target.genre_id = stale.genre_id
    WHEN MATCHED THEN DELETE
""")

# Record applied snapshots, including movies whose valid genre array was empty.
eligible_genre_snapshots_df.select(
    "movie_id",
    F.lit("genres").alias("collection_name"),
    "source_extracted_at",
    "source_batch_id",
    F.current_timestamp().alias("load_timestamp"),
).createOrReplaceTempView("genre_snapshot_state_source")

spark.sql(f"""
    MERGE INTO {state_table} AS target
    USING genre_snapshot_state_source AS source
    ON target.movie_id = source.movie_id
       AND target.collection_name = source.collection_name

    WHEN MATCHED AND
        source.source_extracted_at > target.source_extracted_at
    THEN UPDATE SET
        target.source_extracted_at = source.source_extracted_at,
        target.source_batch_id = source.source_batch_id,
        target.load_timestamp = source.load_timestamp

    WHEN NOT MATCHED THEN INSERT (
        movie_id,
        collection_name,
        source_extracted_at,
        source_batch_id,
        load_timestamp
    ) VALUES (
        source.movie_id,
        source.collection_name,
        source.source_extracted_at,
        source.source_batch_id,
        source.load_timestamp
    )
""")

print("Genre rows inserted:", rows_inserted)
print("Genre rows updated:", rows_updated)
print("Stale genre rows deleted:", stale_genres_df.count())
print(
    "Genre rows now stored for this batch:",
    spark.table(genres_table)
    .filter(F.col("source_batch_id") == batch_id)
    .count(),
)

In [0]:
spark.sql("""
SELECT run_id, status, rows_inserted, rows_updated, rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%silver_movie_genres%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)